# ATML PA3 Task 1: Kaggle Runner
**Repository:** https://github.com/moizejaz-31/ATML-PA3 | **Seed:** `6304`

Every number this notebook produces comes from a real run. If a step fails, the notebook stops with the error.

### Settings
- **Accelerator:** GPU T4 x2. Both GPUs train at first; once only one run is left, the other GPU benchmarks finished models.
- **Internet:** ON.
- **Run with:** Save Version -> Save & Run All (Commit), so it survives closing the browser.

### First run
Set `SMOKE_TEST_ONLY = True` in the first cell and run it once. It takes minutes and tests every step on the real model with a GPU, including stopping and resuming. Then set it to `False` for the real run.

### Multiple sessions
A session cannot finish all of Task 1. Each session stops itself before the time limit and saves its state to `/kaggle/working/task1_outputs`.

To continue: open the notebook, Add Input -> Your Work -> the previous version of this notebook, then Save & Run All again. It finds the previous outputs, resumes unfinished runs from their exact position, and skips finished work.

The last cell prints what is done and what is left. When everything is done, download `task1_results.zip`, unzip it into `task1_llm_finetuning/` in your repository, and commit it. Checkpoints stay out of git. Then fill in `results/qualitative/annotations_42.csv` by hand and re-run `run_task1_figures.py`.

In [ ]:
# 1. Settings and repository
import glob, json, os, shutil, subprocess, threading, time
from queue import Queue, Empty

SMOKE_TEST_ONLY = True          # True: quick test of every step. False: the real experiments.
BRANCH = "fix/task1-pipeline"   # Change to "master" once the branch is merged
SESSION_HOURS = 11.0            # Kaggle stops GPU sessions at about 12 hours; keep a margin
SESSION_START = time.time()

CODE = "/tmp/ATML-PA3"
OUT = "/kaggle/working/task1_outputs"
CKPT = f"{OUT}/checkpoints"
RESULTS = f"{OUT}/results"
IDS = f"{OUT}/splits/metamath_ids_seed6304.json"
CONFIGS = "task1_llm_finetuning/configs"

# Never run the two Full FT variants together: each recovery checkpoint is about 7 GB
TRAINING_PLAN = [["lora_r4", "lora_r16"], ["oplora_r16", "full_ft"], ["full_ft_l2sp"]]
CONDITION_STATE = {
    "baseline": None,
    "lora_r4_1250": "lora_r4/adapters_update_1250.pt",
    "lora_r16_1250": "lora_r16/adapters_update_1250.pt",
    "oplora_r16_1250": "oplora_r16/adapters_update_1250.pt",
    "full_ft_625": "full_ft/model_update_625.pt",
    "full_ft_1250": "full_ft/model_update_1250.pt",
    "full_ft_l2sp_1250": "full_ft_l2sp/model_update_1250.pt",
}

def minutes_left():
    return SESSION_HOURS * 60 - (time.time() - SESSION_START) / 60

def run(command, gpu=None, timeout_min=None, label=""):
    # Streams output and raises on failure, unlike !python. Returns False only on a timeout.
    env = dict(os.environ, PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    if gpu is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(gpu)
    process = subprocess.Popen(command, shell=True, cwd=CODE, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    timed_out = threading.Event()
    timer = None
    if timeout_min is not None:
        def stop():
            timed_out.set()
            process.terminate()
        timer = threading.Timer(max(timeout_min, 0) * 60, stop)
        timer.start()
    prefix = f"[{label}] " if label else ""
    for line in process.stdout:
        print(prefix + line, end="", flush=True)
    code = process.wait()
    if timer:
        timer.cancel()
    if timed_out.is_set():
        print(f"{prefix}Stopped at the session time limit; it continues next session.")
        return False
    if code != 0:
        raise RuntimeError(f"Exit code {code}: {command}")
    return True

def run_on_gpus(jobs, stop_at_session_limit=False):
    # jobs: list of (label, command). Each GPU takes the next job when it is free.
    import torch
    queue, errors = Queue(), []
    for job in jobs:
        queue.put(job)
    def worker(gpu):
        while not errors:
            try:
                label, command = queue.get_nowait()
            except Empty:
                return
            try:
                run(command, gpu=gpu, label=label, timeout_min=minutes_left() - 3 if stop_at_session_limit else None)
            except Exception as error:
                errors.append(error)
    threads = [threading.Thread(target=worker, args=(g,)) for g in range(max(1, torch.cuda.device_count()))]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    if errors:
        raise errors[0]

def is_complete(run_name):
    path = f"{RESULTS}/training/{run_name}/summary.json"
    if not os.path.exists(path):
        return False
    with open(path) as f:
        return json.load(f)["completed"]

shutil.rmtree(CODE, ignore_errors=True)
subprocess.run(["git", "clone", "-q", "-b", BRANCH, "https://github.com/moizejaz-31/ATML-PA3.git", CODE], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# 2. Install dependencies
run('pip install -q -U "transformers>=4.51" datasets accelerate "lm-eval==0.4.13" pyyaml seaborn pytest')
run('pip list 2>/dev/null | grep -iE "^(torch|transformers|datasets|accelerate|lm.eval) "')

In [ ]:
# 3. Restore outputs from a previous session, if one is attached as input
previous = sorted(glob.glob("/kaggle/input/**/task1_outputs", recursive=True))
if previous:
    print(f"Restoring from {previous[-1]}")
    shutil.copytree(previous[-1], OUT, dirs_exist_ok=True)
else:
    print("No previous session attached: starting fresh.")
for path in [CKPT, RESULTS, os.path.dirname(IDS)]:
    os.makedirs(path, exist_ok=True)

In [ ]:
# 4. Hardware check
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Set Accelerator to GPU T4 x2 in the sidebar.")
for i in range(torch.cuda.device_count()):
    prop = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {prop.name} | {prop.total_memory / 1e9:.1f} GB")
print(f"PyTorch {torch.__version__}")

In [ ]:
# 5. Unit tests, then eligible-example selection and the probe set
run("python -m pytest task1_llm_finetuning/tests -q -p no:cacheprovider")
run(f"python -m task1_llm_finetuning.src.data --seed 6304 --ids_path {IDS}")

In [ ]:
# 6. Smoke test on the real model and real data. Writes only to /tmp. Scores here mean nothing.
if SMOKE_TEST_ONLY:
    smoke = "/tmp/task1_smoke"
    shutil.rmtree(smoke, ignore_errors=True)
    start = time.time()
    for name in ["lora_r16", "oplora_r16", "full_ft_l2sp"]:
        train = f"python -m task1_llm_finetuning.src.train --config {CONFIGS}/{name}.yaml --debug --save_dir {smoke}/ckpt --results_dir {smoke}/results/training"
        run(f"{train} --time_limit_min 0", gpu=0, label=f"stop {name}")
        run(train, gpu=0, label=f"resume {name}")

    # Short runs stand in for finished ones, so that the later steps have states to load
    stand_ins = {"lora_r4/adapters": "lora_r16_debug/adapters_update_10", "lora_r16/adapters": "lora_r16_debug/adapters_update_10",
                 "oplora_r16/adapters": "oplora_r16_debug/adapters_update_10", "full_ft/model": "full_ft_l2sp_debug/model_update_10",
                 "full_ft_l2sp/model": "full_ft_l2sp_debug/model_update_10"}
    for target, source in stand_ins.items():
        os.makedirs(os.path.dirname(f"{smoke}/ckpt/{target}"), exist_ok=True)
        shutil.copy(f"{smoke}/ckpt/{source}.pt", f"{smoke}/ckpt/{target}_update_1250.pt")
    shutil.copy(f"{smoke}/ckpt/full_ft/model_update_1250.pt", f"{smoke}/ckpt/full_ft/model_update_625.pt")

    for condition in ["baseline", "oplora_r16_1250", "full_ft_1250"]:
        run(f"python -m task1_llm_finetuning.src.evaluate --condition {condition} --checkpoint_dir {smoke}/ckpt --output_dir {smoke}/results/eval --limit 8", gpu=0, label=f"eval {condition}")
    with open(f"{smoke}/results/eval/baseline/gsm8k_cot_samples.jsonl") as f:
        sample = json.loads(f.readline())
    print("GSM8K prompt:", repr(sample["prompt"]))
    print("GSM8K output:", repr(sample["output"][:300]))
    if "<think>\n\n</think>" not in sample["prompt"]:
        raise RuntimeError("GSM8K prompt does not use the non-thinking chat template")

    run(f"python -m task1_llm_finetuning.src.diagnostics --checkpoint_dir {smoke}/ckpt --output_dir {smoke}/results", gpu=0, label="diagnostics")
    run(f"python -m task1_llm_finetuning.src.qualitative --checkpoint_dir {smoke}/ckpt --output_dir {smoke}/results/qualitative --conditions baseline --limit 2", gpu=0, label="qualitative")
    run(f"python task1_llm_finetuning/run_task1_figures.py --results_dir {smoke}/results", label="figures")
    run(f"du -sh {smoke}/ckpt/*")
    shutil.rmtree(smoke)
    print(f"\nSMOKE TEST PASSED in {(time.time() - start) / 60:.1f} min. Set SMOKE_TEST_ONLY = False for the real run.")

In [ ]:
# 7. Training and benchmarks together. A GPU that is not training evaluates finished models meanwhile.
# A failed benchmark is reported at the end and never stops training.
EVALUATION_RESERVE_MIN = 60
TRAIN_ORDER = ["lora_r4", "lora_r16", "oplora_r16", "full_ft", "full_ft_l2sp"]
FULL_FT_RUNS = {"full_ft", "full_ft_l2sp"}   # Never together: each recovery checkpoint is about 7 GB

def condition_ready(condition):
    state = CONDITION_STATE[condition]
    return state is None or os.path.exists(f"{CKPT}/{state}")

def condition_done(condition):
    return len(glob.glob(f"{RESULTS}/eval/{condition}/*_scores.json")) == 4

def schedule(workers):
    lock = threading.Lock()
    active, failed, errors = set(), set(), []
    training_stopped = threading.Event()

    def next_job():
        with lock:
            if not training_stopped.is_set() and minutes_left() > EVALUATION_RESERVE_MIN + 10:
                for name in TRAIN_ORDER:
                    if name in active or name in failed or is_complete(name):
                        continue
                    if name in FULL_FT_RUNS and active & FULL_FT_RUNS:
                        continue
                    active.add(name)
                    return "train", name
            if minutes_left() > 10:
                for condition in CONDITION_STATE:
                    if condition not in active and condition not in failed and condition_ready(condition) and not condition_done(condition):
                        active.add(condition)
                        return "eval", condition
            return None

    def training_active():
        with lock:
            return bool(active & set(TRAIN_ORDER))

    def worker(gpu):
        while True:
            job = next_job()
            if job is None:
                if training_active():
                    time.sleep(60)   # More models will be ready once training finishes
                    continue
                return
            kind, name = job
            try:
                if kind == "train":
                    limit = minutes_left() - EVALUATION_RESERVE_MIN
                    run(f"python -m task1_llm_finetuning.src.train --config {CONFIGS}/{name}.yaml --save_dir {CKPT} --results_dir {RESULTS}/training --ids_path {IDS} --time_limit_min {limit:.0f}", gpu=gpu, label=name)
                    if not is_complete(name):
                        training_stopped.set()   # Stopped at the time limit: the rest waits for the next session
                else:
                    run(f"python -m task1_llm_finetuning.src.evaluate --condition {name} --checkpoint_dir {CKPT} --output_dir {RESULTS}/eval", gpu=gpu, label=f"eval {name}", timeout_min=minutes_left() - 3)
            except Exception as error:
                print(f"[{name}] FAILED: {error}", flush=True)
                with lock:
                    failed.add(name)
                    errors.append(error)
            finally:
                with lock:
                    active.discard(name)

    threads = [threading.Thread(target=worker, args=(gpu,)) for gpu in range(workers)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    return errors

if not SMOKE_TEST_ONLY:
    errors = schedule(max(1, torch.cuda.device_count()))
    for condition in CONDITION_STATE:
        print(f"benchmarks {condition}: {'done' if condition_done(condition) else 'NOT done'}")
    if errors:
        raise errors[0]

In [ ]:
# 8. Diagnostics and qualitative responses, once all five runs have finished
if not SMOKE_TEST_ONLY:
    if all(is_complete(name) for group in TRAINING_PLAN for name in group):
        if not os.path.exists(f"{RESULTS}/activation_diagnostics.csv"):
            run(f"python -m task1_llm_finetuning.src.diagnostics --checkpoint_dir {CKPT} --output_dir {RESULTS}", gpu=0, timeout_min=minutes_left() - 2)
        run(f"python -m task1_llm_finetuning.src.qualitative --checkpoint_dir {CKPT} --output_dir {RESULTS}/qualitative", gpu=0, timeout_min=minutes_left() - 2)
    else:
        print("Waiting for all five training runs.")

In [ ]:
# 9. Tables, figures, status, and the results archive to commit
if not SMOKE_TEST_ONLY:
    from IPython.display import Image, display
    run(f"python task1_llm_finetuning/run_task1_figures.py --results_dir {RESULTS}")
    for path in sorted(glob.glob(f"{RESULTS}/figures/*.png")):
        print(os.path.basename(path))
        display(Image(filename=path, width=850))
    subprocess.run(f"rm -f /kaggle/working/task1_results.zip && cd {OUT} && zip -q -r /kaggle/working/task1_results.zip results splits", shell=True, check=True)
    for group in TRAINING_PLAN:
        for name in group:
            print(f"{name}: {'finished' if is_complete(name) else 'NOT finished'}")
    print(f"Session used {(time.time() - SESSION_START) / 3600:.1f} h")
    run(f"du -sh {OUT}")